# Issue #1117: Dedup Challenges with Agents SDK + LiteLLM + OpenInference

This notebook reproduces the duplicate-span scenario described in Pylon #1117 and then verifies a `should_export_span` based fix.

**Stack under test:**
- OpenAI Agents SDK with `LitellmModel` (OpenInference instrumentation for `openai-agents`)
- Direct `litellm.completion` calls outside of `Runner.run` (LiteLLM's `langfuse_otel` callback)
- Langfuse Python SDK v4 as the OTEL hub

**How to use:** Run Section 1 (setup), then Section 2 (no filter). Restart the kernel, re-run Section 1, then run Section 3 (with filter). Compare the two sets of trace URLs in the Langfuse UI.

## Section 1: Setup

Installs, credentials, and instrumentation. Run this once per kernel session.

In [ ]:
%pip install -q "langfuse>=4.5.0" "openai-agents[litellm]>=0.0.12" "openinference-instrumentation-openai-agents>=0.1.10" "litellm>=1.79.0"

In [ ]:
import os

os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...")
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...")
os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")

os.environ.setdefault("OPENAI_API_KEY", "sk-...")

assert not os.environ["LANGFUSE_PUBLIC_KEY"].endswith("..."), "Set LANGFUSE_PUBLIC_KEY"
assert not os.environ["LANGFUSE_SECRET_KEY"].endswith("..."), "Set LANGFUSE_SECRET_KEY"
assert not os.environ["OPENAI_API_KEY"].endswith("..."), "Set OPENAI_API_KEY"

In [ ]:
import uuid

RUN_TAG = f"issue-1117-{uuid.uuid4().hex[:8]}"
print("Run tag:", RUN_TAG)

## Section 2: Without the workaround (expect duplicates)

This shows the raw problem: both OpenInference's `openai-agents` processor and LiteLLM's `langfuse_otel` callback emit spans for the same `LitellmModel` call.

**Run Section 1 first.** Then run this section end-to-end.

In [ ]:
from langfuse import get_client

langfuse = get_client()
assert langfuse.auth_check(), "Langfuse credentials invalid"
print("Langfuse initialized (no filter)")

In [ ]:
import litellm
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

OpenAIAgentsInstrumentor().instrument()

if "langfuse_otel" not in litellm.callbacks:
    litellm.callbacks.append("langfuse_otel")

print("OpenInference (openai-agents) + LiteLLM langfuse_otel enabled")

In [ ]:
with langfuse.start_as_current_span(name="direct-litellm-call") as span:
    span.update_trace(tags=[RUN_TAG, "no-filter", "direct-litellm"])
    resp = litellm.completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Say hi in 5 words."}],
    )
    print(resp.choices[0].message.content)
    direct_trace_id = langfuse.get_current_trace_id()

print("Direct LiteLLM trace:", f"{os.environ['LANGFUSE_HOST']}/trace/{direct_trace_id}")

In [ ]:
import asyncio
from agents import Agent, Runner
from agents.extensions.models.litellm_model import LitellmModel

agent = Agent(
    name="assistant",
    instructions="Answer in one short sentence.",
    model=LitellmModel(model="gpt-4o-mini"),
)

with langfuse.start_as_current_span(name="agents-sdk-litellm-run") as span:
    span.update_trace(tags=[RUN_TAG, "no-filter", "agents-litellm"])
    result = await Runner.run(agent, input="What is 2+2?")
    print(result.final_output)
    agents_trace_id = langfuse.get_current_trace_id()

print("Agents+LitellmModel trace:", f"{os.environ['LANGFUSE_HOST']}/trace/{agents_trace_id}")

In [ ]:
langfuse.flush()
print("Flushed.\n")
print("Inspect these traces in Langfuse:")
print(f"  1. Direct LiteLLM:          {os.environ['LANGFUSE_HOST']}/trace/{direct_trace_id}")
print(f"  2. Agents SDK + LitellmModel: {os.environ['LANGFUSE_HOST']}/trace/{agents_trace_id}")
print(f"\nFilter by tag `{RUN_TAG}` to see both side by side.")
print("\nExpected problems:")
print("  - Direct call: may see a `raw_gen_ai_request` child span duplicating the primary LLM span")
print("  - Agents run:  two LLM spans per model call (OpenInference generation + langfuse_otel)")

---

## Section 3: With the workaround (`should_export_span` filter)

**Restart the kernel, re-run Section 1, then run this section.**

The filter below replaces both pieces of David's workaround:

1. Custom `TracingProcessor` wrapping `OpenInferenceTracingProcessor` that drops `GenerationSpanData` → replaced by filtering spans named `"generation"` in the `openinference.instrumentation.openai_agents` scope. `ResponseSpanData` (span name `"response"`) is preserved as the sole emitter for `OpenAIResponsesModel` paths.
2. Monkey-patch of `litellm.integrations.opentelemetry.OpenTelemetry._maybe_log_raw_request` → replaced by filtering spans named `"raw_gen_ai_request"` in the `litellm` scope.

Net result: `langfuse_otel` is the sole emitter for `LitellmModel` and direct LiteLLM calls; OpenInference is the sole emitter for everything else in the Agents SDK (responses, agent/handoff/tool spans).

In [ ]:
from langfuse import Langfuse
from langfuse.span_filter import is_default_export_span

OI_AGENTS_SCOPE = "openinference.instrumentation.openai_agents"

def should_export_span(span):
    if not is_default_export_span(span):
        return False

    scope = span.instrumentation_scope.name if span.instrumentation_scope else ""

    # Drop OpenInference's GenerationSpanData (duplicates langfuse_otel for LitellmModel).
    # ResponseSpanData (span name "response") is kept for OpenAIResponsesModel paths.
    if scope == OI_AGENTS_SCOPE and span.name == "generation":
        return False

    # Drop LiteLLM's raw_gen_ai_request child (duplicates the primary LLM span in same scope).
    if scope.startswith("litellm") and span.name == "raw_gen_ai_request":
        return False

    return True

langfuse = Langfuse(should_export_span=should_export_span)
assert langfuse.auth_check(), "Langfuse credentials invalid"
print("Langfuse initialized WITH should_export_span filter")

In [ ]:
import litellm
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

OpenAIAgentsInstrumentor().instrument()

if "langfuse_otel" not in litellm.callbacks:
    litellm.callbacks.append("langfuse_otel")

print("OpenInference (openai-agents) + LiteLLM langfuse_otel enabled")

In [ ]:
with langfuse.start_as_current_span(name="direct-litellm-call") as span:
    span.update_trace(tags=[RUN_TAG, "with-filter", "direct-litellm"])
    resp = litellm.completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Say hi in 5 words."}],
    )
    print(resp.choices[0].message.content)
    direct_trace_id = langfuse.get_current_trace_id()

print("Direct LiteLLM trace:", f"{os.environ['LANGFUSE_HOST']}/trace/{direct_trace_id}")

In [ ]:
from agents import Agent, Runner
from agents.extensions.models.litellm_model import LitellmModel

agent = Agent(
    name="assistant",
    instructions="Answer in one short sentence.",
    model=LitellmModel(model="gpt-4o-mini"),
)

with langfuse.start_as_current_span(name="agents-sdk-litellm-run") as span:
    span.update_trace(tags=[RUN_TAG, "with-filter", "agents-litellm"])
    result = await Runner.run(agent, input="What is 2+2?")
    print(result.final_output)
    agents_trace_id = langfuse.get_current_trace_id()

print("Agents+LitellmModel trace:", f"{os.environ['LANGFUSE_HOST']}/trace/{agents_trace_id}")

In [ ]:
langfuse.flush()
print("Flushed.\n")
print("Inspect these traces in Langfuse:")
print(f"  1. Direct LiteLLM:          {os.environ['LANGFUSE_HOST']}/trace/{direct_trace_id}")
print(f"  2. Agents SDK + LitellmModel: {os.environ['LANGFUSE_HOST']}/trace/{agents_trace_id}")
print(f"\nFilter by tag `{RUN_TAG}` to see both side by side.")
print("\nExpected result:")
print("  - Direct call: a single primary LLM span, no raw_gen_ai_request child")
print("  - Agents run:  one LLM span per model call (from langfuse_otel), plus agent/response spans from OpenInference")